## Tuning 

tuning boosting models to check if there is any improvement in performace

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

df=pd.read_csv("/home/student/Projects/Startup-outcome-intelligence/data/processed/startup_data_final.csv")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
average_precision_score
)


In [2]:
x=df.drop(columns=["label"])
y=df["label"]

In [11]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=42)

In [4]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# 1 = success (common), 0 = fail (rare), so this weight comes out BELOW 1
spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", spw)

xgb = XGBClassifier(
    n_estimators=300,        # build 300 small trees, one after another
    learning_rate=0.03,      # each tree only gets a tiny vote
    max_depth=3,             # each tree asks at most 3 questions in a row
    min_child_weight=5,      # a branch must cover enough data, no rules about 2 weirdos
    subsample=0.8,           # each tree sees a random 80% of rows
    colsample_bytree=0.8,    # each tree sees a random 80% of columns
    reg_lambda=5,            # penalty for extreme predictions
    gamma=1,                 # only split if it helps by at least this much
    scale_pos_weight=spw,    
    eval_metric="logloss",
    random_state=42,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# P(success) for every train row, from models that never saw that row
oof_success = cross_val_predict(xgb, x_train, y_train, cv=cv, method="predict_proba")[:, 1]

scale_pos_weight: 0.12169366413778962


In [5]:
from sklearn.metrics import precision_recall_curve

p_fail_oof = 1 - oof_success

prec, rec, thr = precision_recall_curve(1 - y_train, p_fail_oof)
f1 = 2 * prec * rec / (prec + rec + 0.000000001) # 0.000000001 so that 0 division doesnt happen

i = np.argmax(f1[:-1])
best = thr[i] #we note this

print("best threshold:", best)
print("failure precision:", prec[i], "| failure recall:", rec[i], "| F1:", f1[i])

best threshold: 0.605087
failure precision: 0.24032586558044808 | failure recall: 0.4970513900589722 | F1: 0.32399780296533054


In [6]:
xgb.fit(x_train, y_train)

p_fail_test = 1 - xgb.predict_proba(x_test)[:, 1]
y_pred_test = (p_fail_test >= best).astype(int)   # 1 = predicted FAIL

print(classification_report(1 - y_test, y_pred_test, target_names=["success", "fail"]))
print(confusion_matrix(1 - y_test, y_pred_test))
print("ROC-AUC:", roc_auc_score(1 - y_test, p_fail_test))
print("Failure PR-AUC:", average_precision_score(1 - y_test, p_fail_test))

              precision    recall  f1-score   support

     success       0.93      0.81      0.86      2439
        fail       0.23      0.48      0.31       297

    accuracy                           0.77      2736
   macro avg       0.58      0.64      0.59      2736
weighted avg       0.85      0.77      0.80      2736

[[1970  469]
 [ 155  142]]
ROC-AUC: 0.7370852159700048
Failure PR-AUC: 0.2582646186199394


## XGBoost with Imbalance Handling and Threshold Tuning

The baseline XGBoost predicted almost every startup as successful (failure recall 1%). To fix this, three changes were made:

- `scale_pos_weight` was set to `n_fail / n_success` (≈ 0.12). Since `label=1` is the majority class (success), the weight is below 1, which makes the model pay more attention to failures.
- Hyperparameters were regularized to reduce overfitting on noisy data: `max_depth=3`, `learning_rate=0.03`, `min_child_weight=5`, `subsample=0.8`, `colsample_bytree=0.8`, `reg_lambda=5`, `gamma=1`, `n_estimators=300`.
- The decision threshold was tuned instead of using the default 0.5. Out-of-fold predictions from 5-fold stratified cross-validation on the training set were used to pick the cutoff that maximized failure F1 (best threshold ≈ 0.605 on P(fail)). The test set was only used once for the final evaluation.

Failure is treated as the positive class in the results below.

### Results (Test Set)

| Metric | Baseline XGBoost | Tuned XGBoost |
|----------|----------|----------|
| Failure Recall | 0.010 | 0.48 |
| Failure Precision | – | 0.23 |
| Failure F1 | 0.020 | 0.31 |
| ROC-AUC | 0.734 | 0.737 |
| Failure PR-AUC | – | 0.258 |
| Accuracy | 0.891 | 0.77 |

Confusion matrix (rows = actual, columns = predicted, order: success, fail):

| | Pred. Success | Pred. Fail |
|----------|----------|----------|
| **Actual Success** | 1970 | 469 |
| **Actual Fail** | 155 | 142 |

### Observations

- Failure recall improved from 1% to 48%, so the model now detects nearly half of the failed startups instead of ignoring them.
- Accuracy dropped from 0.89 to 0.77. This is expected, because the baseline's high accuracy came from predicting "success" for almost everything.
- Failure precision is 0.23, roughly 2x the base failure rate (~11%). Flagged startups are about twice as likely to fail as a randomly chosen one, though many flags are still false alarms.
- Out-of-fold results on the training set (precision 0.24, recall 0.50) closely match the test results (precision 0.23, recall 0.48), which shows the threshold generalizes and was not overfit.
- ROC-AUC barely changed (0.734 to 0.737). Class weighting and threshold tuning change how the model acts on its scores, not how well it ranks startups. The ranking ability appears limited by the information in the current features.
- Further gains from hyperparameter tuning alone are likely to be small. Feature engineering and feature importance analysis are the next steps.

### Next Steps

- Apply the same imbalance handling and threshold tuning to LightGBM for a fair comparison.
- Inspect feature importance to find which columns carry the signal and to check for possible leakage.
- Engineer new features (funding per round, time between rounds, investor-related ratios).

In [7]:
from lightgbm import LGBMClassifier

# same idea as XGB: 1 = success (common), so this weight is BELOW 1
spw = (y_train == 0).sum() / (y_train == 1).sum()

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    num_leaves=7,            # keep below 2^max_depth (=8)
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,        # LightGBM ignores subsample unless this is set
    colsample_bytree=0.8,
    reg_lambda=5,
    scale_pos_weight=spw,
    random_state=42,
    verbose=-1,              # silences the warning spam
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_success_lgbm = cross_val_predict(lgbm, x_train, y_train, cv=cv, method="predict_proba")[:, 1]

In [8]:
p_fail_oof_lgbm = 1 - oof_success_lgbm

prec, rec, thr = precision_recall_curve(1 - y_train, p_fail_oof_lgbm)
f1 = 2 * prec * rec / (prec + rec + 1e-9)

i = np.argmax(f1[:-1])
best_lgbm = thr[i]

print("best threshold:", best_lgbm)
print("failure precision:", prec[i], "| failure recall:", rec[i], "| F1:", f1[i])

best threshold: 0.5988616488808525
failure precision: 0.2405866032500991 | failure recall: 0.5113732097725358 | F1: 0.3272237192413887


In [9]:
lgbm.fit(x_train, y_train)

p_fail_test_lgbm = 1 - lgbm.predict_proba(x_test)[:, 1]
y_pred_test_lgbm = (p_fail_test_lgbm >= best_lgbm).astype(int)   # 1 = predicted FAIL

print(classification_report(1 - y_test, y_pred_test_lgbm, target_names=["success", "fail"]))
print(confusion_matrix(1 - y_test, y_pred_test_lgbm))
print("ROC-AUC:", roc_auc_score(1 - y_test, p_fail_test_lgbm))
print("Failure PR-AUC:", average_precision_score(1 - y_test, p_fail_test_lgbm))

              precision    recall  f1-score   support

     success       0.93      0.80      0.86      2439
        fail       0.24      0.50      0.32       297

    accuracy                           0.77      2736
   macro avg       0.58      0.65      0.59      2736
weighted avg       0.85      0.77      0.80      2736

[[1956  483]
 [ 148  149]]
ROC-AUC: 0.7395528332387702
Failure PR-AUC: 0.2653684353423855


## LightGBM with Imbalance Handling and Threshold Tuning

The same procedure used for XGBoost was applied to LightGBM so the two models can be compared fairly:

- `scale_pos_weight` was set to `n_fail / n_success` (≈ 0.12). Since `label=1` is the majority class (success), the weight is below 1, which makes the model pay more attention to failures.
- Hyperparameters were regularized to reduce overfitting on noisy data: `max_depth=3`, `num_leaves=7`, `learning_rate=0.03`, `min_child_samples=20`, `subsample=0.8` (with `subsample_freq=1`), `colsample_bytree=0.8`, `reg_lambda=5`, `n_estimators=300`.
- The decision threshold was tuned using out-of-fold predictions from 5-fold stratified cross-validation on the training set, picking the cutoff that maximized failure F1 (best threshold ≈ 0.599 on P(fail)). The test set was only used once for the final evaluation.

Failure is treated as the positive class in the results below.

### Results (Test Set)

| Metric | Baseline LightGBM | Tuned LightGBM |
|----------|----------|----------|
| Failure Recall | 0.017 | 0.50 |
| Failure Precision | – | 0.24 |
| Failure F1 | 0.032 | 0.32 |
| ROC-AUC | 0.742 | 0.740 |
| Failure PR-AUC | – | 0.265 |
| Accuracy | 0.891 | 0.77 |

Confusion matrix (rows = actual, columns = predicted, order: success, fail):

| | Pred. Success | Pred. Fail |
|----------|----------|----------|
| **Actual Success** | 1956 | 483 |
| **Actual Fail** | 148 | 149 |

### Observations

- Failure recall improved from 1.7% to 50%, so the model now detects half of the failed startups instead of ignoring them.
- Accuracy dropped from 0.89 to 0.77, which is expected because the baseline's high accuracy came from predicting "success" for almost everything.
- Failure precision is 0.24, roughly 2x the base failure rate (~11%), so flagged startups are about twice as likely to fail as a random one, though many flags are still false alarms.
- Out-of-fold results on the training set (precision 0.24, recall 0.51) closely match the test results (precision 0.24, recall 0.50), showing the threshold generalizes and was not overfit.
- ROC-AUC stayed essentially unchanged from the baseline (0.742 to 0.740). Class weighting and threshold tuning change how the model acts on its scores, not how well it ranks startups.

### Comparison with Tuned XGBoost

| Metric | XGBoost | LightGBM |
|----------|----------|----------|
| Failure Recall | 0.48 | 0.50 |
| Failure Precision | 0.23 | 0.24 |
| Failure F1 | 0.31 | 0.32 |
| ROC-AUC | 0.737 | 0.740 |
| Failure PR-AUC | 0.258 | 0.265 |
| Failures caught (of 297) | 142 | 149 |

- LightGBM performed marginally better on every metric, but the differences are small and likely within noise for a test set of this size.
- Two different boosting libraries, tuned the same way, reach nearly identical performance (ROC-AUC ≈ 0.74, PR-AUC ≈ 0.26). This suggests the limiting factor is the information in the current features rather than the choice of model.
- Further hyperparameter tuning is unlikely to give large gains. Feature importance analysis and feature engineering are the next steps.

In [10]:
joblib.dump({"model": lgbm, "threshold": best_lgbm, "columns": list(x_train.columns)},"../models/lgbm_model.joblib")

['../models/lgbm_model.joblib']